In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import pandas as pd

In [31]:
df = pd.read_csv("https://raw.githubusercontent.com/kzming2007/Catholic_ML_final_project/refs/heads/main/dataset/ur3_cobotops.csv")

# REGRESSION
## current,speed 데이터를 기반으로 tool current를 예측

In [35]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd

# 1. 결측치 없는 클린 데이터 로드 및 분리
df_clean = df.dropna().copy()

# 입력 피처(X): 12개 조인트 피처
input_cols = [
    'Current_J0', 'Current_J1', 'Current_J2', 'Current_J3', 'Current_J4', 'Current_J5',
    'Speed_J0', 'Speed_J1', 'Speed_J2', 'Speed_J3', 'Speed_J4', 'Speed_J5'
]
X = df_clean[input_cols]

# 타겟(y): 그리퍼 전류
y = df_clean['Tool_current']

# 2. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. 비교할 스케일러 정의
scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler()
}

print("--- [최적 피벗] 그리퍼 전류(Tool_current) 스케일러별 회귀 결과 ---")

# 4. 루프를 돌며 스케일러별 학습 및 평가
for name, scaler in scalers.items():
    # 데이터 스케일링
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 4. 다항 피처 변환 (Degree=2) - 최적의 성능을 낸 비결!
    poly = PolynomialFeatures(degree=2, include_bias=False)
    X_train_poly = poly.fit_transform(X_train_scaled)
    X_test_poly = poly.transform(X_test_scaled)

    # 모델 학습 및 예측
    model = LinearRegression()
    model.fit(X_train_poly, y_train)
    y_pred = model.predict(X_test_poly)

    # 결과 출력
    print(f"\n[{name} 결과]")
    print(f"R² (결정계수): {r2_score(y_test, y_pred):.4f}")
    print(f"MSE (평균제곱오차): {mean_squared_error(y_test, y_pred):.6f}")


--- [최적 피벗] 그리퍼 전류(Tool_current) 스케일러별 회귀 결과 ---

[StandardScaler 결과]
R² (결정계수): 0.2340
MSE (평균제곱오차): 0.004207

[MinMaxScaler 결과]
R² (결정계수): 0.2340
MSE (평균제곱오차): 0.004207


In [37]:
# [코랩 새 블록] Ridge, Lasso, ElasticNet 모델 스케일러별 비교
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd

# 1. 스케일러 및 모델 정의
scalers = {
    "Standard": StandardScaler(),
    "MinMax": MinMaxScaler()
}

models = {
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1.0),
    "ElasticNet": ElasticNet(alpha=1.0, l1_ratio=0.5)
}

# 결과를 저장할 리스트
results = []

# 2. 루프 실행 (스케일러 x 모델 조합)
for s_name, scaler in scalers.items():
    # 현재 스케일러로 데이터 변환
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

        # 4. 다항 피처 변환 (Degree=2) - 최적의 성능을 낸 비결!
    poly = PolynomialFeatures(degree=2, include_bias=False)
    X_train_poly = poly.fit_transform(X_train_scaled)
    X_test_poly = poly.transform(X_test_scaled)

    for m_name, model in models.items():
        # 모델 학습 및 예측
        model.fit(X_train_poly, y_train)
        y_pred = model.predict(X_test_poly)

        # 평가지표 계산
        r2 = r2_score(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)

        # 결과 저장
        results.append({
            "Scaler": s_name,
            "Model": m_name,
            "R2": round(r2, 4),
            "MSE": round(mse, 6)
        })

# 3. 데이터프레임 변환 및 출력
df_res = pd.DataFrame(results)
print("--- [최적 피벗] 선형 규제 모델 성능 비교 결과 ---")
display(df_res.sort_values(by="R2", ascending=False)) # R2 기준 내림차순 정렬


--- [최적 피벗] 선형 규제 모델 성능 비교 결과 ---


,Scaler,Model,R2,MSE
0,Standard,Ridge,0.2340,0.004207
3,MinMax,Ridge,0.1760,0.004525
1,Standard,Lasso,-0.0023,0.005504
2,Standard,ElasticNet,-0.0023,0.005504
4,MinMax,Lasso,-0.0023,0.005504
5,MinMax,ElasticNet,-0.0023,0.005504
